In [ ]:
!pip install plotly pandas requests nbformat -q

import requests
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta

API_URL = 'https://djtnqbvkhqftmtnsityx.supabase.co/rest/v1/'
API_KEY = 'sb_publishable_NnOjc1iJWmA7j418h79mEg_AHh9h49u'
HEADERS = {'apikey': API_KEY, 'Authorization': f'Bearer {API_KEY}'}

def query_supabase(endpoint, select='*', filters=None, limit=10000):
    """Query Supabase REST API with filters."""
    params = {}
    if select: params['select'] = select
    if filters:
        for k, v in filters.items():
            params[k] = f'eq.{v}'
    if limit: params['limit'] = limit
    r = requests.get(f'{API_URL}{endpoint}', headers=HEADERS, params=params)
    if r.status_code == 200:
        return pd.DataFrame(r.json())
    else:
        print(f'Error {r.status_code}: {r.text[:200]}')
        return pd.DataFrame()

In [ ]:
# Query daily funding for crypto symbols — last 12 months
one_year_ago = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')

df = query_supabase(
    'daily_funding',
    select='venue,symbol,date,avg_rate_bps,asset_class',
    filters={'asset_class': 'crypto'},
    limit=50000
)

df['date'] = pd.to_datetime(df['date'])
df = df[df['date'] >= one_year_ago].copy()
df['avg_rate_bps'] = pd.to_numeric(df['avg_rate_bps'], errors='coerce')
df = df.dropna(subset=['avg_rate_bps'])

print(f"Loaded {len(df):,} rows across {df['venue'].nunique()} venues and {df['symbol'].nunique()} symbols")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
df.head()

In [ ]:
# Line chart: avg funding rate over time by venue
daily_avg = df.groupby(['date', 'venue'])['avg_rate_bps'].mean().reset_index()

fig = px.line(
    daily_avg, x='date', y='avg_rate_bps', color='venue',
    title='Crypto Funding Rates by Venue (Daily Avg)',
    labels={'avg_rate_bps': 'Avg Rate (bps)', 'date': 'Date', 'venue': 'Venue'},
    template='plotly_dark'
)
fig.update_layout(height=500, hovermode='x unified')
fig.show()

In [ ]:
# Histogram: distribution of funding rates by venue
fig = px.histogram(
    df, x='avg_rate_bps', color='venue', barmode='overlay', nbins=80,
    title='Funding Rate Distribution by Venue',
    labels={'avg_rate_bps': 'Avg Rate (bps)', 'venue': 'Venue'},
    template='plotly_dark', opacity=0.6
)
fig.update_layout(height=450)
fig.show()

In [ ]:
# Summary statistics by venue
summary = df.groupby('venue')['avg_rate_bps'].agg(
    count='count', mean='mean', median='median',
    min='min', max='max', std='std'
).round(3).sort_values('mean', ascending=False)

summary.style.format({
    'mean': '{:.2f}', 'median': '{:.2f}', 'min': '{:.2f}',
    'max': '{:.2f}', 'std': '{:.2f}'
}).set_caption('Funding Rate Summary (bps) by Venue')